# Les solveurs formels DÉCIDENT — calcul propositionnel, premier ordre, modal

**Notebook pédagogique CoursIA** · Track CI-3 (#1466, Epic #1448) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Ce notebook illustre le **cœur symbolique** : des solveurs formels qui rendent un verdict **tranché**
(satisfiable / non-satisfiable, consistant / inconsistant) sur de petites théories logiques. Il tourne
**firsthand** (cellules exécutées, sorties réelles) contre le code local `argumentation_analysis` :
PySAT (PL), EProver + Tweety (FOL), SimpleMlReasoner + SPASS (modal).

> **Privacy.** Le scénario est entièrement **synthétique et domaine-public** : un club d'échecs
organise sa sortie annuelle (tournoi, buffet, cotisations, météo du jour J). Aucun contenu de corpus,
aucun `raw_text`, aucune donnée nominative réelle.

> **Anti-théâtre.** Les verdicts ci-dessous sont calculés par de vrais solveurs au moment de
l'exécution — jamais codés en dur. Si un solveur n'est pas disponible dans l'environnement, la
cellule le signale honnêtement (*dégradé*) au lieu d'afficher un faux vert.

In [1]:
# Imports + configuration. On démarre la JVM (requise par FOL + modal ; le PL pur-PySAT n'en a pas besoin).
import logging, os, sys

logging.disable(logging.INFO)   # sorties propres, on garde seulement les verdicts

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    sys.path.insert(0, _cwd)
    os.chdir(_cwd)

from argumentation_analysis.core.jvm_setup import initialize_jvm
from argumentation_analysis.agents.core.logic.tweety_bridge import TweetyBridge
from argumentation_analysis.core.config import settings, SolverChoice, ModalSolverChoice

_JVM_OK = initialize_jvm()
_BRIDGE = TweetyBridge() if _JVM_OK else None
print(f"JVM démarrée : {_JVM_OK}")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-07-15 14:44:24 [WARNING] [Services.CryptoService] crypto_service.__init__:46 - Service de chiffrement initialisé sans clé. Le chiffrement est désactivé.
JVM démarrée : True


## §1 — Logique propositionnelle (PL) : le plan du jour J

Le club décide de l'organisation avec des **atomes** booléens :
- `T` = un tournoi est organisé
- `B` = un buffet est prévu
- `R` = un repas (le buffet compte comme repas : `B => R`)

On encode les contraintes comme des formules propositionnelles et on demande au solveur SAT
([cadical195](https://github.com/arminbiere/cadical), via PySAT) si l'ensemble est **satisfiable**
(il existe une valuation des atomes qui rend toutes les formules vraies).

| Cas | Contraintes | Attendu |
|-----|-------------|---------|
| **Plan tenable** | `T => B` (tournoi ⇒ buffet), `B => R` (buffet ⇒ repas), `T | B` (au moins l'un des deux) | **SAT** (ex. T=B=R=Vrai) |
| **Plan impossible** | `T => B`, `T`, `!B` (tournoi organisé, mais pas de buffet) | **UNSAT** (T et T⇒B forcent B, contredit !B) |

In [2]:
from argumentation_analysis.agents.core.logic.sat_handler import SATHandler

_pl = SATHandler()   # backend cadical195 (PySAT)

# Cas A : plan tenable — il existe une organisation cohérente.
PLAN_TENABLE = ["T => B", "B => R", "T | B"]
sat_ok, msg_ok = _pl.check_consistency(PLAN_TENABLE)

# Cas B : plan impossible — le tournoi est organisé sans buffet, pourtant T => B l'exige.
PLAN_IMPOSSIBLE = ["T => B", "T", "!B"]
sat_no, msg_no = _pl.check_consistency(PLAN_IMPOSSIBLE)

# Query d'entailment : T => B et T entraînent-ils logiquement B ?
entails_B = _pl.query(["T => B", "T"], "B")

print("PLAN_TENABLE   :", PLAN_TENABLE)
print(f"  -> {'SAT (plan tenable)' if sat_ok else 'UNSAT'} | {msg_ok}")
print()
print("PLAN_IMPOSSIBLE:", PLAN_IMPOSSIBLE)
print(f"  -> {'SAT' if sat_no else 'UNSAT (plan impossible)'} | {msg_no}")
print()
print(f"Entailment : (T => B) ∧ T  ⊨  B  ?  {entails_B}")

PLAN_TENABLE   : ['T => B', 'B => R', 'T | B']
  -> SAT (plan tenable) | Consistent (SAT). Model: {'T': True, 'B': True, 'R': True}

PLAN_IMPOSSIBLE: ['T => B', 'T', '!B']
  -> UNSAT (plan impossible) | Inconsistent (UNSAT). Solver: cadical195

Entailment : (T => B) ∧ T  ⊨  B  ?  True


## §2 — Logique du premier ordre (FOL) : « tout adhérent inscrit paie sa cotisation »

La FOL quantifie sur des **individus** (les adhérents du club). On déclare un *sort* `clubmember`
et des prédicats `Registers(X)` / `Pays(X)`, puis on exprime la règle universelle :
`∀X (Registers(X) => Pays(X))`.

Deux théories, deux verdicts genuines rendus par **EProver** (prouveur externe `E 2.0`) **et**
recoupés par **Tweety SimpleFolReasoner** (prouveur en-JVM) :

- **Théorie cohérente** : la règle + Alice inscrite ET payée → **consistante**.
- **Théorie incohérente** : la règle + Bob inscrit mais `!Pays(bob)` → **inconsistante**
  (Bob inscrit implique Bob paie, contredit `!Pays(bob)`).

In [3]:
# FOL — deux backends (EProver + Tweety) recoupent le verdict.
FOL_CONSISTENT = (
    "clubmember = {alice, bob}\n"
    "type(Pays(clubmember))\n"
    "type(Registers(clubmember))\n"
    "\n"
    "forall X: (Registers(X) => Pays(X))\n"
    "Registers(alice)\n"
    "Pays(alice)\n"
)
FOL_INCONSISTENT = (
    "clubmember = {bob}\n"
    "type(Pays(clubmember))\n"
    "type(Registers(clubmember))\n"
    "\n"
    "forall X: (Registers(X) => Pays(X))\n"
    "Registers(bob)\n"
    "!Pays(bob)\n"
)

print("Règle : ∀X (Registers(X) => Pays(X))\n")
for _label, _kb in [("Cohérente (alice payée)", FOL_CONSISTENT),
                    ("Incohérente (bob impayé)", FOL_INCONSISTENT)]:
    print(f"-- Théorie {_label} --")
    for _solver, _choice in [("EProver", SolverChoice.EPROVER), ("Tweety", SolverChoice.TWEETY)]:
        settings.solver = _choice
        ok, msg = _BRIDGE.check_consistency(_kb, "first_order")
        print(f"   {_solver:8s}: {'CONSISTANTE' if ok else 'INCONSISTANTE':15s} ({msg})")
    print()

Règle : ∀X (Registers(X) => Pays(X))

-- Théorie Cohérente (alice payée) --
   EProver : CONSISTANTE     (FOL consistency check (EProver): consistent)
   Tweety  : CONSISTANTE     (FOL consistency check (SimpleFolReasoner): consistent)

-- Théorie Incohérente (bob impayé) --
   EProver : INCONSISTANTE   (FOL consistency check (EProver): inconsistent)
   Tweety  : INCONSISTANTE   (FOL consistency check (SimpleFolReasoner): inconsistent)



## §3 — Logique modale : nécessité et possibilité (météo du jour J)

La logique modale ajoute `□` (nécessairement) et `◇` (possiblement). On encode la règle du club :
« **nécessairement**, s'il pleut alors la pelouse est mouillée » (`[](Rain => Wet)`), puis on teste
la **consistance modale**.

- **Consistante** : `[](Rain => Wet)` + `Rain` → pas de contradiction.
- **Inconsistante** : `Rain` + `!Rain` → une contradiction nue.

Deux backends : **Tweety SimpleMlReasoner** (énumération de mondes de Kripke, en-JVM) **et**
**SPASS** (prouveur par saturation, `ext_tools/spass/SPASS.exe`).

In [4]:
# Modal — deux backends (Tweety + SPASS) sous la logique K.
MOD_CONSISTENT   = "type(Rain)\ntype(Wet)\n\n[](Rain => Wet)\nRain\n"
MOD_INCONSISTENT = "type(Rain)\n\nRain\n!Rain\n"

def _modal(backends):
    """Run each modal backend; report honest-degraded if a backend can't decide."""
    out = {}
    for _name, _solver, _prefer in backends:
        settings.modal_solver = _solver
        settings.modal_prefer_spass_when_available = _prefer
        row = {}
        for _lab, _kb in [("consistante", MOD_CONSISTENT), ("inconsistante", MOD_INCONSISTENT)]:
            try:
                ok, msg = _BRIDGE.check_consistency(_kb, "K")
                if ok is None:
                    row[_lab] = "DÉGRADÉ (solveur indisponible)"
                else:
                    row[_lab] = "CONSISTANTE" if ok else "INCONSISTANTE"
            except Exception as _e:
                row[_lab] = f"DÉGRADÉ ({type(_e).__name__})"
        out[_name] = row
    return out

_modal_backends = [
    ("Tweety", ModalSolverChoice.TWEETY, False),   # pure-Java, toujours disponible
    ("SPASS",  ModalSolverChoice.SPASS,  True),    # binaire externe (peut manquer sur d'autres machines)
]
_modal_res = _modal(_modal_backends)
print("KB consistante   : [](Rain => Wet) ∧ Rain")
print("KB inconsistante : Rain ∧ !Rain\n")
for _name, _row in _modal_res.items():
    print(f"  {_name:8s}: consistante -> {_row['consistante']:15s} | inconsistante -> {_row['inconsistante']}")

KB consistante   : [](Rain => Wet) ∧ Rain
KB inconsistante : Rain ∧ !Rain

  Tweety  : consistante -> CONSISTANTE     | inconsistante -> INCONSISTANTE
  SPASS   : consistante -> CONSISTANTE     | inconsistante -> INCONSISTANTE


## §4 — Bonus : cadre de Dung (argumentation abstraite)

Les formalismes précédents portent sur des *formules*. La **théorie de l'argumentation de Dung**
porte sur des *arguments* abstraits reliés par une relation d'attaque : un ensemble d'arguments
`S` est une **extension préférée** s'il est (i) sans conflit interne (aucun argument de `S` n'en
attaque un autre) et (ii) il se défend contre toute attaque (tout attaquant est lui-même attaqué
par `S`).

Scénario : trois arguments sur la sortie du club s'attaquent mutuellement —
`A` (faire le tournoi) attaque `B` (annuler, fatigue), `B` attaque `C` (buffet seul, démotivant),
`C` attaque `B` en retour. On calcule l'extension préférée en pur Python (algorithme naïf,
transparent).

In [5]:
# Dung : extension préférée en pur Python (algorithme naïf, transparent).
AF_ARGS = ["A_tournoi", "B_annuler", "C_buffet_seul"]
AF_ATTACKS = [("A_tournoi", "B_annuler"), ("B_annuler", "C_buffet_seul"), ("C_buffet_seul", "B_annuler")]

def _attacked_by_set(s, attacks):
    """Arguments attaqués par au moins un membre de s."""
    return {b for (a, b) in attacks if a in s}

def _is_conflict_free(s, attacks):
    return not _attacked_by_set(s, attacks) & s

def _defends(s, args, attacks):
    """s défend ses membres : tout attaquant d'un membre est attaqué par s."""
    for x in s:
        for (a, b) in attacks:
            if b == x and a not in _attacked_by_set(s, attacks):
                return False
    return True

def _is_admissible(s, args, attacks):
    return _is_conflict_free(s, attacks) and _defends(s, args, attacks)

import itertools
def _preferred_extensions(args, attacks):
    """Extensions préférées : admissibles et maximales pour l'inclusion."""
    adm = []
    for k in range(len(args) + 1):
        for combo in itertools.combinations(args, k):
            s = set(combo)
            if _is_admissible(s, args, attacks):
                if not any(s < other for other in adm):
                    adm.append(s)
    # ne garder que les maximaux
    prefs = [s for s in adm if not any(s < t for t in adm)]
    return [sorted(s) for s in prefs]

_prefs = _preferred_extensions(AF_ARGS, AF_ATTACKS)
print("Attaques :", [f"{a}→{b}" for a, b in AF_ATTACKS])
print("Extensions préférées :", _prefs)
print("(A n'est attaqué par personne → il survit dans toute extension préférée.)")

Attaques : ['A_tournoi→B_annuler', 'B_annuler→C_buffet_seul', 'C_buffet_seul→B_annuler']
Extensions préférées : [['A_tournoi', 'C_buffet_seul']]
(A n'est attaqué par personne → il survit dans toute extension préférée.)


## §5 — Synthèse : le symbole tranche

Chaque formalisme capture un **type de raisonnement distinct**, et chacun rend un verdict **binaire,
calculé firsthand** par un solveur réel :

- **PL** décide si un ensemble de contraintes booléennes *peut être satisfait* (SAT/UNSAT).
- **FOL** décide si une théorie quantifiée est *cohérente* (consistante/inconsistante),
  recoupée par deux prouveurs indépendants.
- **Modal** décide la cohérence sous *nécessité/possibilité*, recoupée par deux prouveurs.
- **Dung** décide quels arguments *survivent* à l'attaque mutuelle (extension préférée).

Le point pédagogique : en logique formelle, **la syntaxe n'est pas décorative** — elle engage un
solveur qui tranche. `!Pays(bob)` contredit `∀X (Registers(X) => Pays(X))` non pas par opinion, mais
par démonstration automatique (EProver). C'est ce que l'on entend par *décidabilité computationnelle*
du raisonnement.

In [6]:
# Tableau récapitulatif des verdicts firsthand.
print("=" * 64)
print(" FORMEL   |  QUESTION                         |  VERDICT")
print("-" * 64)
print(f" PL       |  plan tenable                    |  {'SAT' if sat_ok else 'UNSAT'}")
print(f" PL       |  plan impossible                 |  {'SAT' if sat_no else 'UNSAT'}")
print(f" PL       |  (T=>B) ∧ T  ⊨  B               |  {entails_B}")
settings.solver = SolverChoice.EPROVER
_fol_c, _ = _BRIDGE.check_consistency(FOL_CONSISTENT, "first_order")
_fol_i, _ = _BRIDGE.check_consistency(FOL_INCONSISTENT, "first_order")
print(f" FOL      |  règle + alice payée             |  {'CONSISTANTE' if _fol_c else 'INCONSISTANTE'}")
print(f" FOL      |  règle + bob impayé              |  {'CONSISTANTE' if _fol_i else 'INCONSISTANTE'}")
settings.modal_solver = ModalSolverChoice.TWEETY
settings.modal_prefer_spass_when_available = False
_mod_c, _ = _BRIDGE.check_consistency(MOD_CONSISTENT, "K")
_mod_i, _ = _BRIDGE.check_consistency(MOD_INCONSISTENT, "K")
print(f" MODAL    |  [](Rain=>Wet) ∧ Rain            |  {'CONSISTANTE' if _mod_c else 'INCONSISTANTE'}")
print(f" MODAL    |  Rain ∧ !Rain                    |  {'CONSISTANTE' if _mod_i else 'INCONSISTANTE'}")
print(f" DUNG     |  extension préférée              |  {_prefs}")
print("=" * 64)
print("Tous les verdicts ci-dessus sont calculés par de vrais solveurs au moment de l'exécution.")

 FORMEL   |  QUESTION                         |  VERDICT
----------------------------------------------------------------
 PL       |  plan tenable                    |  SAT
 PL       |  plan impossible                 |  UNSAT
 PL       |  (T=>B) ∧ T  ⊨  B               |  True
 FOL      |  règle + alice payée             |  CONSISTANTE
 FOL      |  règle + bob impayé              |  INCONSISTANTE
 MODAL    |  [](Rain=>Wet) ∧ Rain            |  CONSISTANTE
 MODAL    |  Rain ∧ !Rain                    |  INCONSISTANTE
 DUNG     |  extension préférée              |  [['A_tournoi', 'C_buffet_seul']]
Tous les verdicts ci-dessus sont calculés par de vrais solveurs au moment de l'exécution.
